[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_9_deep_q_learning.ipynb)

<div style="text-align:center">
    <h1>
        Deep Q-Learning
    </h1>
</div>

<br><br>

<div style="text-align:center">

In this notebook, we extend the Q-Learning algorithm to use function approximators (Neural Networks). The resulting algorithm is known as Deep Q-Learning.
</div>



<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_stats, seed_everything


## Import the necessary software libraries:

In [ ]:
import random
import copy
import gym
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch import nn as nn
from torch.optim import AdamW
from tqdm import tqdm

## Create and prepare the environment

### Create the environment

**Your turn.** In the code cell below, create the CartPole environment, seed it for reproducibility, reset it, and render one frame so you can see it.

Hints:
- Gym environments are created with `gym.make(...)`. The classic balancing task is `'CartPole-v0'`.
- The course helper `seed_everything(...)` makes runs repeatable.
- Render with `env.render(mode='rgb_array')` and display with matplotlib.

**Next:** read the state and action sizes from the environment.

The Q-network will need to know how many numbers describe a state (the input size) and how many actions exist (the output size). Look at `env.observation_space.shape` and `env.action_space.n`. Store them in variables you can reuse later.

### Prepare the environment to work with PyTorch

In [ ]:
class PreprocessEnv(gym.Wrapper):

    def __init__(self, env):
        gym.Wrapper.__init__(self, env)

    def reset(self):
        obs = self.env.reset()
        return torch.from_numpy(obs).unsqueeze(dim=0).float()

    def step(self, action):
        action = action.item()
        next_state, reward, done, info = self.env.step(action)
        next_state = torch.from_numpy(next_state).unsqueeze(dim=0).float()
        reward = torch.tensor(reward).view(1, -1).float()
        done = torch.tensor(done).view(1, -1)
        return next_state, reward, done, info

In [ ]:
env = PreprocessEnv(env)

In [ ]:
state = env.reset()
action = torch.tensor(0)
next_state, reward, done, _ = env.step(action)
print(f"Sample state: {state}")
print(f"Next state: {next_state}, Reward: {reward}, Done: {done}")

## Create the Q-Network and policy

<br><br>

### Create the Q-Network: $\hat q(s,a\,|\,\theta)$

**Your turn.** Build the Q-network in the cell below.

Remember the idea of Deep Q-Learning: instead of a lookup table, a **neural network** estimates the value of each action from the state. So the network should:
- take a state vector as input (size = number of state dimensions),
- pass it through a couple of hidden layers with non-linear activations,
- output **one value per action**.

`nn.Sequential`, `nn.Linear`, and `nn.ReLU` are the building blocks you need. Do **not** put a Softmax at the end -- these are values, not probabilities.

### Create the target Q-Network: $\hat q(s, a\,|\,\theta_{targ})$

**Your turn.** Create the *target network* in the cell below.

Why we need it: training is unstable if the target we chase moves every time we update the network. The fix is a **frozen copy** of the Q-network that we only refresh every few episodes. Think about how to make an exact, independent copy and put it in evaluation mode.

### Create the exploratory policy: $b(s)$

In [ ]:
def policy(state, epsilon=0.):
    if torch.rand(1) < epsilon:
        return torch.randint(num_actions, (1, 1))
    else:
        av = q_network(state).detach()
        return torch.argmax(av, dim=-1, keepdim=True)

## Create the Experience Replay buffer

<br>
<div style="text-align:center">
    <p>A simple buffer that stores transitions of arbitrary values, adapted from
    <a href="https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html#training">this source.</a></p>
</div>


In [ ]:
class ReplayMemory:

    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.memory = []
        self.position = 0

    def insert(self, transition):
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        assert self.can_sample(batch_size)

        batch = random.sample(self.memory, batch_size)
        batch = zip(*batch)
        return [torch.cat(items) for items in batch]

    def can_sample(self, batch_size):
        return len(self.memory) >= batch_size * 10

    def __len__(self):
        return len(self.memory)

## Implement the algorithm

**Your turn.** Write the `deep_q_learning` training loop in the cell below. Pull together every piece you have built.

A checklist for the loop:
1. Create an optimiser for the Q-network and an empty replay buffer.
2. For each episode, play until `done`:
   - choose an action with the $\epsilon$-greedy `policy`,
   - step the environment and **store** the transition in the buffer.
3. Once the buffer has enough data, sample a batch and:
   - get the Q-values for the actions actually taken,
   - build the **TD target** $r + \gamma\,\max_{a'} \hat q_{targ}(s', a')$ using the **target network** (remember to zero the future term when the episode ended),
   - take an MSE-loss step.
4. Every few episodes, copy the live weights into the target network.

Keep track of the loss and the episode return so you can plot them afterwards.

**Finally:** call your `deep_q_learning` function to train the agent (a few hundred episodes is enough), and store the returned statistics in a variable named `stats` so the plotting cell below works.

## Show results

### Plot execution stats

In [ ]:
plot_stats(stats)

### Test the resulting agent

In [ ]:
test_agent(env, policy, episodes=2)

## Resources

[[1] Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf)